# SankhyaVox — Baseline Model Training

Self-contained notebook for training and evaluating baseline classifiers
(GMM, k-NN+DTW, SVM) on SankhyaVox processed data.

**Prerequisites:** Run `DataPipeline().build()` first to generate `data_processed/`
and the category CSV files (`human.csv`, `tts.csv`).

Then import the `data_processed/` folder into this notebook environment.

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Config

Set `DATA_PROCESSED` to the path where your `data_processed/` folder is located.

In [ ]:
# ── Adjust this path to where data_processed/ is located ──
DATA_PROCESSED = Path("../data_processed")

# ── Checkpoint save directory ──
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ── Baseline model hyperparameters ──
SVM_C_RANGE     = [0.1, 1, 10, 100]
SVM_GAMMA_RANGE = [1e-4, 1e-3, 1e-2, 0.1]
KNN_K           = 5
GMM_COMPONENTS  = 3

## SankhyaVoxDataset (self-contained)

Copy of the dataset class — loads from CSV, no project imports needed.

In [ ]:
class SankhyaVoxDataset:
    """
    Unified dataset for SankhyaVox training and evaluation.

    Reads ``<processed_dir>/human.csv``, ``tts.csv``, ``augmented.csv``
    and concatenates them into a single DataFrame.

    Parameters
    ----------
    processed_dir : Path or str, optional
        Root of the processed data tree.  Default: ``config.PROCESSED_DIR``.
    categories : list of str, optional
        Which categories to load.  Default: all CSVs present on disk.

    ``ds[i]`` returns::

        {
            "audio_path":   str,      # path to source .wav segment
            "audio_source": str,      # "human" | "tts" | "augmented"
            "speaker_id":   str,      # e.g. "S01"
            "token":        str,      # Sanskrit label, e.g. "eka"
            "label":        int,      # numeric value, e.g. 1
            "feature":      ndarray,  # shape (n_frames, 39)
        }
    """

    CATEGORIES = ("human", "tts", "augmented")

    def __init__(
        self,
        processed_dir: Optional[Path] = None,
        categories: Optional[List[str]] = None,
    ):
        self._root = Path(processed_dir) if processed_dir else PROCESSED_DIR
        self._categories = categories or list(self.CATEGORIES)
        self._df = self._load_csvs()

    # ── Loading ───────────────────────────────────────────────────────────

    def _load_csvs(self) -> pd.DataFrame:
        """Load and concatenate category CSV files."""
        frames = []
        for cat in self._categories:
            csv_path = self._root / f"{cat}.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                df["category"] = cat
                frames.append(df)

        if not frames:
            return pd.DataFrame()

        return pd.concat(frames, ignore_index=True)

    # ── Properties ────────────────────────────────────────────────────────

    @property
    def df(self) -> pd.DataFrame:
        """Full metadata DataFrame."""
        return self._df

    @property
    def speakers(self) -> List[str]:
        """Sorted list of unique speaker IDs."""
        if self._df.empty:
            return []
        return sorted(self._df["speaker"].unique().tolist())

    @property
    def tokens(self) -> List[str]:
        """Sorted list of unique Sanskrit token labels."""
        if self._df.empty:
            return []
        return sorted(self._df["sanskrit_label"].unique().tolist())

    # ── Indexing ──────────────────────────────────────────────────────────

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """Return a sample dict for the given global index."""
        row = self._df.iloc[idx]
        return {
            "audio_path": row["wav_path"],
            "audio_source": row["category"],
            "speaker_id": row["speaker"],
            "token": row["sanskrit_label"],
            "label": int(row["label"]),
            "feature": np.load(row["npy_path"]),
        }

    def __len__(self) -> int:
        return len(self._df)

    def __iter__(self) -> Iterator[Dict[str, Any]]:
        for i in range(len(self)):
            yield self[i]

    # ── Batch access ──────────────────────────────────────────────────────

    def get_Xy(self) -> Tuple[List[np.ndarray], List[int]]:
        """Return ``(features_list, labels_list)`` for model training."""
        X, y = [], []
        for i in range(len(self)):
            s = self[i]
            X.append(s["feature"])
            y.append(s["label"])
        return X, y

    # ── Filtering / Splitting ─────────────────────────────────────────────

    def filter(
        self,
        category: Optional[str] = None,
        speaker: Optional[str] = None,
        label: Optional[int] = None,
    ) -> "SankhyaVoxDataset":
        """Return a new dataset filtered by the given criteria."""
        mask = pd.Series(True, index=self._df.index)
        if category is not None:
            mask &= self._df["category"] == category
        if speaker is not None:
            mask &= self._df["speaker"] == speaker
        if label is not None:
            mask &= self._df["label"] == label

        filtered = SankhyaVoxDataset.__new__(SankhyaVoxDataset)
        filtered._root = self._root
        filtered._categories = self._categories
        filtered._df = self._df[mask].reset_index(drop=True)
        return filtered

    def split_by_speakers(
        self,
        train: List[str],
        val: List[str],
        test: List[str],
    ) -> Tuple["SankhyaVoxDataset", "SankhyaVoxDataset", "SankhyaVoxDataset"]:
        """Split dataset into train / val / test by speaker IDs."""

        def _subset(spk_list: List[str]) -> "SankhyaVoxDataset":
            ds = SankhyaVoxDataset.__new__(SankhyaVoxDataset)
            ds._root = self._root
            ds._categories = self._categories
            ds._df = self._df[self._df["speaker"].isin(spk_list)].reset_index(
                drop=True
            )
            return ds

        return _subset(train), _subset(val), _subset(test)

    # ── Display ───────────────────────────────────────────────────────────

    def __repr__(self) -> str:
        cats = self._df["category"].value_counts().to_dict() if len(self._df) else {}
        cat_str = ", ".join(f"{k}={v}" for k, v in sorted(cats.items()))
        return f"SankhyaVoxDataset(samples={len(self)}, {cat_str})"

    def summary(self) -> str:
        """Return a formatted summary of the dataset contents."""
        lines = [repr(self)]
        if not self._df.empty:
            lines.append(f"  Speakers: {', '.join(self.speakers)}")
            lines.append(f"  Tokens:   {', '.join(self.tokens)}")
            for cat in self.CATEGORIES:
                sub = self._df[self._df["category"] == cat]
                if not sub.empty:
                    lines.append(
                        f"  [{cat}] {len(sub)} samples, "
                        f"{sub['speaker'].nunique()} speakers"
                    )
        return "\n".join(lines)


## Load Dataset

In [ ]:
ds = SankhyaVoxDataset(DATA_PROCESSED)
print(repr(ds))
print(f"Speakers: {ds.speakers}")
print(f"Tokens:   {ds.tokens}")
ds.df.head()

## Train / Test Split (by speaker)

**Training:** human + TTS speakers combined for maximum data.  
**Testing:** human speakers only (real speech evaluation).

In [ ]:
# ── Human speakers (used for test set) ──
human_ds = ds.filter(category="human")
human_speakers = human_ds.speakers
print(f"Human speakers: {human_speakers}")

# Hold out last human speaker for testing
test_spk = [human_speakers[-1]]
train_human_spk = human_speakers[:-1]

print(f"Train (human): {train_human_spk}")
print(f"Test  (human): {test_spk}")

# ── Training set: human train speakers + ALL TTS speakers ──
ds_train_human, _, ds_test = human_ds.split_by_speakers(train_human_spk, [], test_spk)
ds_tts = ds.filter(category="tts")

# Combine human train + TTS into one training set
X_train_h, y_train_h = ds_train_human.get_Xy()
X_train_t, y_train_t = ds_tts.get_Xy() if len(ds_tts) > 0 else ([], [])
X_train = X_train_h + X_train_t
y_train = y_train_h + y_train_t

# Test set: human only
X_test, y_test = ds_test.get_Xy()

print(f"\nTrain samples: {len(X_train)} (human={len(X_train_h)}, tts={len(X_train_t)})")
print(f"Test samples:  {len(X_test)} (human only)")

---
## GMM Classifier

In [ ]:
class GMMClassifier:
    """
    GMM baseline: one GMM per class, classify by max log-likelihood.

    Parameters
    ----------
    n_components : int
        Number of Gaussian components per class GMM.
    n_classes : int
        Number of target classes.
    checkpoint_path : str or Path, optional
        If given, load a previously saved model.
    """

    def __init__(
        self,
        n_components: int = 3,
        n_classes: int = 13,
        checkpoint_path: Optional[str] = None,
    ):
        self.n_components = n_components
        self.n_classes = n_classes
        self.models: dict[int, GaussianMixture] = {}

        if checkpoint_path:
            self.load(checkpoint_path)

    @staticmethod
    def summarise(features: np.ndarray) -> np.ndarray:
        """Summarise variable-length MFCC to fixed-length vector (mean + std)."""
        return np.concatenate([features.mean(axis=0), features.std(axis=0)])

    def fit(self, X: list[np.ndarray], y: list[int]) -> "GMMClassifier":
        """
        Fit one GMM per class.

        Parameters
        ----------
        X : list of ndarray, each (n_frames, feat_dim)
        y : list of int labels
        """
        summaries: dict[int, list[np.ndarray]] = {}
        for feat, label in zip(X, y):
            summaries.setdefault(label, []).append(self.summarise(feat))

        for label, vecs in summaries.items():
            data = np.array(vecs)
            n_comp = min(self.n_components, len(data))
            gmm = GaussianMixture(n_components=n_comp, covariance_type="diag", random_state=42)
            gmm.fit(data)
            self.models[label] = gmm

        return self

    def predict(self, X: list[np.ndarray]) -> np.ndarray:
        """Predict class labels for a list of feature sequences."""
        preds = []
        for feat in X:
            vec = self.summarise(feat).reshape(1, -1)
            best_label = max(self.models, key=lambda l: self.models[l].score(vec))
            preds.append(best_label)
        return np.array(preds)

    def score(self, X: list[np.ndarray], y: list[int]) -> float:
        """Return accuracy on the given data."""
        preds = self.predict(X)
        return float(np.mean(preds == np.array(y)))

    def save(self, path: str) -> None:
        """Save model to a pickle file."""
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({"n_components": self.n_components, "models": self.models}, f)
        print(f"Saved GMMClassifier -> {path}")

    def load(self, path: str) -> None:
        """Load model from a pickle file."""
        with open(path, "rb") as f:
            data = pickle.load(f)
        self.n_components = data["n_components"]
        self.models = data["models"]
        print(f"Loaded GMMClassifier <- {path}")

In [ ]:
gmm = GMMClassifier(n_components=GMM_COMPONENTS)
gmm.fit(X_train, y_train)

gmm_preds = gmm.predict(X_test)
gmm_acc = accuracy_score(y_test, gmm_preds)
print(f"GMM Accuracy: {gmm_acc:.3f}")
print(classification_report(y_test, gmm_preds, zero_division=0))

gmm.save(str(CHECKPOINT_DIR / "gmm_classifier.pkl"))

---
## k-NN + DTW Classifier

In [ ]:
def _dtw_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Compute DTW distance between two feature sequences."""
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d = np.sum((a[i - 1] - b[j - 1]) ** 2)
            cost[i, j] = d + min(cost[i - 1, j], cost[i, j - 1], cost[i - 1, j - 1])

    return float(np.sqrt(cost[n, m]))


class KNNDTWClassifier:
    """
    k-NN classifier with DTW distance on raw MFCC sequences.

    Parameters
    ----------
    k : int
        Number of neighbours.
    checkpoint_path : str or Path, optional
        If given, load a previously saved model.
    """

    def __init__(
        self,
        k: int = 5,
        checkpoint_path: Optional[str] = None,
    ):
        self.k = k
        self._X: list[np.ndarray] = []
        self._y: list[int] = []

        if checkpoint_path:
            self.load(checkpoint_path)

    def fit(self, X: list[np.ndarray], y: list[int]) -> "KNNDTWClassifier":
        """Store training data (lazy — no model fitting needed)."""
        self._X = list(X)
        self._y = list(y)
        return self

    def predict_one(self, query: np.ndarray) -> int:
        """Predict label for a single feature sequence."""
        dists = [_dtw_distance(query, ref) for ref in self._X]
        idx = np.argsort(dists)[: self.k]
        labels = [self._y[i] for i in idx]
        # Majority vote
        counts: dict[int, int] = {}
        for l in labels:
            counts[l] = counts.get(l, 0) + 1
        return max(counts, key=lambda l: counts[l])

    def predict(self, X: list[np.ndarray]) -> np.ndarray:
        """Predict class labels for a list of feature sequences."""
        return np.array([self.predict_one(x) for x in X])

    def score(self, X: list[np.ndarray], y: list[int]) -> float:
        """Return accuracy on the given data."""
        preds = self.predict(X)
        return float(np.mean(preds == np.array(y)))

    def save(self, path: str) -> None:
        """Save training data to a pickle file."""
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({"k": self.k, "X": self._X, "y": self._y}, f)
        print(f"Saved KNNDTWClassifier -> {path}")

    def load(self, path: str) -> None:
        """Load training data from a pickle file."""
        with open(path, "rb") as f:
            data = pickle.load(f)
        self.k = data["k"]
        self._X = data["X"]
        self._y = data["y"]
        print(f"Loaded KNNDTWClassifier <- {path}")


In [ ]:
knn = KNNDTWClassifier(k=KNN_K)
knn.fit(X_train, y_train)

print(f"Running k-NN+DTW prediction on {len(X_test)} samples (this may take a while)...")
knn_preds = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_preds)
print(f"k-NN+DTW Accuracy: {knn_acc:.3f}")
print(classification_report(y_test, knn_preds, zero_division=0))

knn.save(str(CHECKPOINT_DIR / "knn_dtw_classifier.pkl"))

---
## SVM Classifier

In [ ]:
class SVMClassifier:
    """
    SVM baseline with RBF kernel and optional grid search.

    Parameters
    ----------
    kernel : str
        SVM kernel type.
    C : float, optional
        Regularisation parameter.  If ``None``, will be tuned via grid search.
    gamma : float or str, optional
        Kernel coefficient.  If ``None``, will be tuned via grid search.
    checkpoint_path : str or Path, optional
        If given, load a previously saved model.
    """

    KernelType = Literal["linear", "poly", "rbf", "sigmoid", "precomputed"]

    def __init__(
        self,
        kernel: KernelType = "rbf",
        C: Optional[float] = None,
        gamma: Optional[float] = None,
        checkpoint_path: Optional[str] = None,
    ):
        self.kernel: SVMClassifier.KernelType = kernel
        self.C = C
        self.gamma = gamma
        self.scaler = StandardScaler()
        self.model: Optional[SVC] = None

        if checkpoint_path:
            self.load(checkpoint_path)

    @staticmethod
    def summarise(features: np.ndarray) -> np.ndarray:
        """Summarise variable-length MFCC to fixed-length vector (mean + std)."""
        return np.concatenate([features.mean(axis=0), features.std(axis=0)])

    def fit(
        self,
        X: list[np.ndarray],
        y: list[int],
        grid_search: bool = True,
        cv: int = 3,
    ) -> "SVMClassifier":
        """
        Fit the SVM.

        Parameters
        ----------
        X : list of ndarray, each (n_frames, feat_dim)
        y : list of int labels
        grid_search : bool
            If True and C/gamma are None, run grid search over
            ``config.SVM_C_RANGE`` and ``config.SVM_GAMMA_RANGE``.
        cv : int
            Cross-validation folds for grid search.
        """
        data = np.array([self.summarise(feat) for feat in X])
        labels = np.array(y)

        data = self.scaler.fit_transform(data)

        if grid_search and (self.C is None or self.gamma is None):
            param_grid = {"C": SVM_C_RANGE, "gamma": SVM_GAMMA_RANGE}
            gs = GridSearchCV(
                SVC(kernel=self.kernel),
                param_grid,
                cv=cv,
                scoring="accuracy",
                n_jobs=-1,
            )
            gs.fit(data, labels)
            self.model = gs.best_estimator_
            self.C = gs.best_params_["C"]
            self.gamma = gs.best_params_["gamma"]
            print(f"Grid search best: C={self.C}, gamma={self.gamma}, "
                  f"acc={gs.best_score_:.3f}")
        else:
            self.model = SVC(
                kernel=self.kernel,
                C=self.C or 1.0,
                gamma=self.gamma or "scale",
            )
            self.model.fit(data, labels)

        return self

    def predict(self, X: list[np.ndarray]) -> np.ndarray:
        """Predict class labels for a list of feature sequences."""
        assert self.model is not None, "Model not fitted yet. Call fit() first."
        data = np.array([self.summarise(feat) for feat in X])
        data = self.scaler.transform(data)
        return self.model.predict(data)

    def score(self, X: list[np.ndarray], y: list[int]) -> float:
        """Return accuracy on the given data."""
        preds = self.predict(X)
        return float(np.mean(preds == np.array(y)))

    def save(self, path: str) -> None:
        """Save model + scaler to a pickle file."""
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({
                "kernel": self.kernel, "C": self.C, "gamma": self.gamma,
                "scaler": self.scaler, "model": self.model,
            }, f)
        print(f"Saved SVMClassifier -> {path}")

    def load(self, path: str) -> None:
        """Load model + scaler from a pickle file."""
        with open(path, "rb") as f:
            data = pickle.load(f)
        self.kernel = data["kernel"]
        self.C = data["C"]
        self.gamma = data["gamma"]
        self.scaler = data["scaler"]
        self.model = data["model"]
        print(f"Loaded SVMClassifier <- {path}")


In [ ]:
svm = SVMClassifier()
svm.fit(X_train, y_train, grid_search=True)

svm_preds = svm.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f"SVM Accuracy: {svm_acc:.3f}")
print(classification_report(y_test, svm_preds, zero_division=0))

svm.save(str(CHECKPOINT_DIR / "svm_classifier.pkl"))

---
## Results Summary

In [ ]:
results = pd.DataFrame({
    "Model": ["GMM", "k-NN+DTW", "SVM (RBF)"],
    "Accuracy": [gmm_acc, knn_acc, svm_acc],
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(results["Model"], results["Accuracy"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy")
ax.set_title("Baseline Model Comparison")
for i, v in enumerate(results["Accuracy"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

## Confusion Matrices

In [ ]:
labels_sorted = sorted(set(y_test))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, name in zip(axes, [gmm_preds, knn_preds, svm_preds], ["GMM", "k-NN+DTW", "SVM"]):
    cm = confusion_matrix(y_test, preds, labels=labels_sorted)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=labels_sorted, yticklabels=labels_sorted)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()